# Shape: generate a domain into ADLS Gen2 tables (Synapse PySpark notebook)

Runs on a Synapse **Apache Spark pool** with the Shape wheel and the `sqllocks-shape-domains`
wheel installed (pool packages or workspace packages; see `integrations/synapse/RUNBOOK.md`).
Shape needs **Python 3.11 or newer**, so the pool must use a runtime with it.

Generates every table of an installed domain at a scale preset with a seed **on the driver**
(`maxRows` refuses a scale too large for it), writes each as a Delta or Parquet folder
`<outputPath>/tables/<tablePrefix><table>`, and writes the contract that the domain's own schema
implies for these tables to `<outputPath>/<domain>/contract.json`.
`shape_profile_domain_synapse` profiles the tables and checks them against that contract.

The exit value is `{domain, scale, seed, mode, tableFormat, tablesPath, tablePrefix, tables,
totalRows, contractPath, kernel}`. `mssparkutils.notebook.exit` is the last statement, outside any
`try`/`except`.


In [ ]:
# Parameters cell (Synapse: toggle "parameter cell"). A pipeline overrides these.
# Paths are abfss://<container>@<account>.dfs.core.windows.net/<path> URLs.
domain = "retail"  # an installed domain (`shape list`)
scale = "small"  # a scale preset of the domain (`shape presets`)
seed = 42  # the same seed always generates the same rows
mode = ""  # "3nf" or "star"; empty keeps the domain's default schema
outputPath = (
    "abfss://shape@<account>.dfs.core.windows.net/generated"  # below: tables/, contract.json
)
tablePrefix = ""  # tables are written as <outputPath>/tables/<tablePrefix><table>
tableFormat = "delta"  # delta | parquet
writeMode = "overwrite"  # overwrite | append
maxRows = 20_000_000  # refuse a scale with more rows: the data is generated on the driver
linkedServiceName = ""  # optional: Synapse linked service that grants access to the storage

In [ ]:
import json
import tempfile
from pathlib import Path

import shape
from shape.integrations.fabric import generation
from shape.kernel.dispatch import get_kernel

try:  # Synapse Spark pools predefine `mssparkutils`; newer runtimes also offer the import
    from notebookutils import mssparkutils
except ImportError:
    pass

KERNEL = get_kernel().NAME  # "rust" with a platform wheel, "python" with the pure-Python wheel
print(f"Shape {shape.__version__}, kernel: {KERNEL}")

MAX_LISTED = 100  # keep the exit value small (well under 1 MB)
tableFormat = str(tableFormat).strip().lower()
if tableFormat not in ("delta", "parquet"):
    raise ValueError(f"tableFormat must be 'delta' or 'parquet', got {tableFormat!r}")
if str(writeMode) not in ("overwrite", "append"):
    raise ValueError(f"writeMode must be 'overwrite' or 'append', got {writeMode!r}")
# Names reach storage paths and table names: only identifiers are accepted.
generation.check_name(str(domain), "domain")
generation.check_name(str(scale), "scale")
if tablePrefix:
    generation.check_name(str(tablePrefix), "tablePrefix")
if not str(outputPath).startswith(("abfss://", "abfs://")):
    raise ValueError("outputPath must be an abfss:// URL")

# Spark writes with the notebook user's identity, or with a linked service's identity.
if linkedServiceName:
    spark.conf.set("spark.storage.synapse.linkedServiceName", linkedServiceName)  # noqa: F821
    spark.conf.set(  # noqa: F821
        "fs.azure.account.oauth.provider.type",
        "com.microsoft.azure.synapse.tokenlibrary.LinkedServiceBasedTokenProvider",
    )

out_root = str(outputPath).rstrip("/")
tables_root = f"{out_root}/tables"
contract_url = f"{out_root}/{domain}/contract.json"

In [ ]:
planned = generation.plan_row_counts(str(domain), str(scale), str(mode) or None)
if sum(planned.values()) > int(maxRows):
    raise ValueError(
        f"{domain!r} at scale {scale!r} is {sum(planned.values()):,} rows, above maxRows="
        f"{int(maxRows):,}: the data is generated on the driver. Use a smaller scale."
    )
result = generation.generate_domain(
    str(domain), scale=str(scale), seed=int(seed), mode=str(mode) or None
)
row_counts = {name: table.num_rows for name, table in result.tables.items()}
print(f"Generated {sum(row_counts.values()):,} rows in {len(row_counts)} tables")

In [ ]:
tables = []
for name in result.generation_order:
    arrow_table = generation.delta_ready(result.tables[name])
    target = f"{tables_root}/{tablePrefix}{name}"
    sdf = spark.createDataFrame(arrow_table.to_pandas())  # noqa: F821
    sdf.write.format(tableFormat).mode(str(writeMode)).save(target)
    tables.append({"table": name, "path": target, "rows": arrow_table.num_rows})
    print(f"  {name:<24} {arrow_table.num_rows:>10,} rows -> {target}")

# The contract the domain's own schema implies for these tables; the profile notebook checks
# the written tables against it.
contract = generation.domain_contract(result.schema, row_counts)
mssparkutils.fs.put(contract_url, json.dumps(contract), True)  # noqa: F821
print("Contract written to", contract_url)

In [ ]:
result_value = {
    "domain": domain,
    "scale": scale,
    "seed": int(seed),
    "mode": str(mode) or "default",
    "tableFormat": tableFormat,
    "tablesPath": tables_root,
    "tablePrefix": str(tablePrefix),
    "tables": tables[:MAX_LISTED],
    "totalRows": sum(t["rows"] for t in tables),
    "contractPath": contract_url,
    "kernel": KERNEL,
}
print(json.dumps(result_value, indent=2)[:4000])

In [ ]:
mssparkutils.notebook.exit(json.dumps(result_value))  # noqa: F821